In [ ]:
# Install only if these packages are not already available in the active kernel.
%pip install -q langchain-community langchain-core langchain-google-genai langchain-text-splitters pypdf faiss-cpu tiktoken

In [ ]:
import os
from pathlib import Path

import pandas as pd
import tiktoken
from IPython.display import display
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = Path("M&A_Due_Diligence_Playbook.pdf")
if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"M&A Playbook not found at {PDF_PATH.resolve()}. Run this notebook from the repository root."
    )

documents = PyPDFLoader(str(PDF_PATH)).load()
page_texts = [doc.page_content or "" for doc in documents]
empty_pages = [i + 1 for i, text in enumerate(page_texts) if not text.strip()]
print(f"Loaded {len(documents)} pages from {PDF_PATH.name}")
display(pd.DataFrame([{
    "pages": len(documents),
    "extracted_characters": sum(map(len, page_texts)),
    "empty_pages": len(empty_pages),
}]))

## Chunking experiment


In [ ]:
tokenizer = tiktoken.get_encoding("cl100k_base")
chunk_experiments = []
for size, overlap in [(400, 60), (600, 100), (800, 120)]:
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=size,
        chunk_overlap=overlap,
        add_start_index=True,
    )
    candidate_chunks = splitter.split_documents(documents)
    token_counts = [len(tokenizer.encode(doc.page_content)) for doc in candidate_chunks]
    chunk_experiments.append({
        "chunk_size": size,
        "overlap": overlap,
        "chunk_count": len(candidate_chunks)
    })
display(pd.DataFrame(chunk_experiments))

CHUNK_SIZE = 600
CHUNK_OVERLAP = 100
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)
chunks = text_splitter.split_documents(documents)
if not chunks:
    raise ValueError("The supplied playbook produced no text chunks.")
print(f"Selected {len(chunks)} chunks; sample metadata: {chunks[0].metadata}")

## Embedding concepts and Gemini setup


In [ ]:
import getpass
from langchain_google_genai import GoogleGenerativeAIEmbeddings

api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Gemini API key (input is hidden): ")
if not api_key or not api_key.strip():
    raise RuntimeError("A Gemini API key is required to build or query the RAG index.")
os.environ["GOOGLE_API_KEY"] = api_key.strip()

CHAT_MODEL = "gemini-3.6-flash"
EMBEDDING_MODEL = "gemini-embedding-2"
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

## Build the FAISS index

In [ ]:
import time

from langchain_community.vectorstores import FAISS

batch_size = 25
vector_store = None

print("Embedding chunks in batches...")
for start in range(0, len(chunks), batch_size):
    batch = chunks[start:start + batch_size]
    if vector_store is None:
        vector_store = FAISS.from_documents(documents=batch, embedding=embeddings)
    else:
        vector_store.add_documents(documents=batch)
    print(f"Indexed chunks {start} to {min(start + batch_size, len(chunks))}...")
    if start + batch_size < len(chunks):
        time.sleep(35)

print(f"FAISS index contains {vector_store.index.ntotal} vectors")

## Grounded question answering

The prompt requires the model to answer only from retrieved playbook excerpts, decline unsupported questions, and cite the page labels provided with the retrieved context.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0)
rag_prompt = ChatPromptTemplate.from_template(
    """You are an M&A knowledge assistant. Use only the retrieved excerpts from the supplied M&A Due Diligence Playbook.
Do not use outside knowledge, fill gaps with assumptions, or claim that the playbook says something not present in the excerpts.
If the answer is not supported by the excerpts, say exactly: "I couldn't find this information in the provided M&A Playbook."
Answer concisely but completely. Cite supporting statements using the page labels included in the excerpts, such as [page 2].

Retrieved excerpts:
{context}

Question: {question}

Answer:"""
)
rag_chain = rag_prompt | llm | StrOutputParser()
FALLBACK_ANSWER = "I couldn't find this information in the provided M&A Playbook."

def answer_question(question: str, k: int = 1) -> dict:
    """Answer from the playbook, returning page labels and retrieved text for review."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Please provide a non-empty question.")
    if not isinstance(k, int) or k < 1:
        raise ValueError("k must be a positive integer.")

    retrieved_docs = vector_store.similarity_search(question.strip(), k=min(k, vector_store.index.ntotal))
    context_parts = []
    source_pages = []
    for doc in retrieved_docs:
        page = doc.metadata.get("page")
        page_label = f"page {page + 1}" if isinstance(page, int) else "page unknown"
        context_parts.append(f"[{page_label}]\n{doc.page_content}")
        if page_label not in source_pages:
            source_pages.append(page_label)
    context = "\n\n".join(context_parts)
    if not context.strip():
        return {
            "question": question.strip(),
            "answer": FALLBACK_ANSWER,
            "sources": [],
            "retrieved_context": [],
        }
    answer = rag_chain.invoke({"context": context, "question": question.strip()})
    return {
        "question": question.strip(),
        "answer": answer,
        "sources": source_pages,
        "retrieved_context": context_parts,
    }


In [ ]:
example_result = answer_question(
    "What should a mutual confidentiality agreement or NDA cover?",
    k=5,
)
print("Question:", example_result["question"])
print("\nAnswer:\n", example_result["answer"])
print("\nRetrieved source pages:", ", ".join(example_result["sources"]) or "none")
print("\nRetrieved context:")
for excerpt in example_result["retrieved_context"]:
    print("\n", excerpt)

## RAG concepts and design answers

- **Why chunk?** Retrieval selects focused pieces rather than passing an entire document for every question; chunks also fit model context limits.
- **If chunks are too small:** a relevant clause may lose the sentence or section that explains its meaning, causing incomplete retrieval.
- **If chunks are too large:** unrelated material can be returned together, similarity becomes less precise, and more context tokens are used.
- **Chunk-size choice:** the experiment reports counts and approximate token sizes for three configurations. A 600-token target with 100-token overlap is a practical middle ground for retaining neighboring context; inspect retrieved excerpts and adjust if answers show missing context.
- **What is an embedding?** A numeric representation of text in a vector space, intended to preserve useful semantic relationships.
- **Why embeddings?** They let the vector store find passages relevant to the meaning of a question, not only exact keyword matches.
- **Semantic similarity:** the vector store ranks chunks by vector distance/similarity to the question embedding; the closest retrieved chunks form the context passed to the answer model.
 correct.